In [1]:
# pandas for data manipulation
# re for regular expressions
import re

# ensure local project packages can be imported (so 'utils' is found)
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

# pyplot for plotting
import matplotlib.pyplot as plt

# numpy for numerical operations
import numpy as np
import pandas as pd

# seaborn for advanced plotting
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, make_scorer

# sklearn for machine learning
from sklearn.model_selection import GridSearchCV, train_test_split

# other .py files
from utils.csv_import import get_csv_files_generalistic, get_csv_files_generalistic_2

# set global numpy seed for reproducibility
np.random.seed(42)

In [2]:
positions = ["z00", "a00", "a01", "a02", "a09", "a10",
        "a11", "b00", "b01", "b02", "b03", "b04", "b05", "b06",
        "b07", "b08", "b09", "b10", "b11", "c01", "c02", "c03",
        "c04", "c05", "c06", "c07", "c08", "c09", "c10", "c11",
        "d01", "d02", "d03", "d04", "d05", "d06", "d07", "d08",
        "d09", "d10", "d11", "e04", "e05", "e06",]

esps_id = [1, 2, 3, 4]  



In [3]:
#path: str = (
#    "C:\\Users\\pedro\\OneDrive - Universidade de Coimbra\\Ambiente de Trabalho"
#    "\\tese\\thesis-project\\data"
#    "\\CSI DATA RENAMED [alterado]"
#)

path: str = (
    "C:\\Users\\Oscar\\Documents\\GitHub\\thesis-project_isac\\data\\CSI DATA DM RENAMED [com w00 alterado]"
)

FileMap = dict[str, dict[str, str]]

print(path)

C:\Users\Oscar\Documents\GitHub\thesis-project_isac\data\CSI DATA DM RENAMED [com w00 alterado]


In [4]:

data_files = get_csv_files_generalistic_2(path)

Found file: C:\Users\Oscar\Documents\GitHub\thesis-project_isac\data\CSI DATA DM RENAMED [com w00 alterado]\00-z00-01-01.csv
Found file: C:\Users\Oscar\Documents\GitHub\thesis-project_isac\data\CSI DATA DM RENAMED [com w00 alterado]\00-z00-01-02.csv
Found file: C:\Users\Oscar\Documents\GitHub\thesis-project_isac\data\CSI DATA DM RENAMED [com w00 alterado]\00-z00-02-01.csv
Found file: C:\Users\Oscar\Documents\GitHub\thesis-project_isac\data\CSI DATA DM RENAMED [com w00 alterado]\00-z00-02-02.csv
Found file: C:\Users\Oscar\Documents\GitHub\thesis-project_isac\data\CSI DATA DM RENAMED [com w00 alterado]\00-z00-03-01.csv
Found file: C:\Users\Oscar\Documents\GitHub\thesis-project_isac\data\CSI DATA DM RENAMED [com w00 alterado]\00-z00-03-02.csv
Found file: C:\Users\Oscar\Documents\GitHub\thesis-project_isac\data\CSI DATA DM RENAMED [com w00 alterado]\00-z00-04-01.csv
Found file: C:\Users\Oscar\Documents\GitHub\thesis-project_isac\data\CSI DATA DM RENAMED [com w00 alterado]\00-z00-04-02.csv


In [21]:
# função que lê valores RSSI do ficheiro CSV - não utilizada [eliminada]

# função que lê e processa os valores CSI do ficheiro CSV
# # semelhante à função de "Wi-Fi-Sensing-[meu].ipynb"
# # mas o ciclo for é usado ao chamar a função e não na função
def read_and_parse_csi(
    file_path: str, num_samples: int = 120, selected_subcarriers: np.ndarray = None
):
    # para cada ficheiro CSV (posição e esp)
    # lemos 120 amostras da coluna 26 - Porquê apenas 120?
    df = pd.read_csv(file_path, header=None)
    len_row = df.shape[0]
    #print(f"Number of rows in the file: {len_row}")
    
    #csi_raw: pd.Series = df.iloc[:num_samples, 26]
    csi_raw: pd.Series = df.iloc[:, 26]
         
    valid_csi: list[list[int]] = []

    # para cada array da lista de arrays
    for entry in csi_raw:
        if pd.isna(entry) or "[" not in str(entry):
            continue

        try:
            # extrai todos os valores inteiros da string
            nums = [int(n) for n in re.findall(r"-?\d+", str(entry))]

            # se o array tiver 128 valores, adiciona à lista de CSI válidos
            if len(nums) == 128:
                valid_csi.append(nums)
        except ValueError:
            continue

    # converter para array numpy
    valid_csi = np.array(valid_csi)
    if valid_csi.shape[0] == 0:
        return np.empty((0, 0)), np.empty((0,))

    # 64 complex subcarriers
    # converter para números complexos
    complex_csi = valid_csi[:, ::2] + 1j * valid_csi[:, 1::2]

    # FFT shift puts DC in the center (index 32)
    fft_csi = np.fft.fftshift(complex_csi, axes=1)

    # Extract the 52 active subcarriers (IEEE 802.11n standard): [6:58]
    active = fft_csi[:, 6:58]

    # Remove subcarriers at positions 25, 26, 27 (center)
    active = np.delete(active, [25, 26, 27], axis=1)

    # selecting specific subcarriers
    subcarrier_indices = [i for i in range(2, 48)]
    selected_subcarriers = np.array(subcarrier_indices)
    if selected_subcarriers is not None: # Este if é sempre True por causa da linha acima
        active = active[:, selected_subcarriers]

    # compute módulo dos valores complexos
    magnitudes = np.abs(active)
    return magnitudes

In [6]:
# A Normalização original era feita através da média e maximo absoluto das magnitudes de cada esp na sala em vazio.
# No entanto, para deteção de presença, o conhecimento da sala em vazio não é possível.

# Assim, a normalização é feita através da média dos segmentos de magnitudes de cada esp em todas as posições.

def normalization(csi_data: dict[str, dict[str, np.ndarray]]):

    normalization_mean: dict[int, np.ndarray] = {}
    normalization_abs_max: dict[int, np.ndarray] = {}

    esp_ids: list[int] = [1, 2, 3, 4] # Manter o 4 aqui para já

    # Get normalization parameters
    for esp_id in esp_ids:
        all_magnitudes = []
        for key in csi_data:
            if f"esp{esp_id}_" in key:
                all_magnitudes.append(csi_data[key])

        all_magnitudes_concat = np.vstack(all_magnitudes)
        normalization_mean[esp_id] = np.mean(all_magnitudes_concat, axis=0)
        normalization_abs_max[esp_id] = np.max(np.abs(all_magnitudes_concat), axis=0)

        csi_data_normalized = csi_data - normalization_mean
        csi_data_normalized = csi_data_normalized / normalization_abs_max


    return csi_data_normalized



In [7]:
# função que segmenta o sinal em janelas de
# tamanho window_size com sobreposição [window - no_overlap]
def segments_signal(signal: np.ndarray, window_size: int = 10, no_overlap: int = 2):
    segments: list[np.ndarray] = []
    t_segment: list[np.ndarray] = []

    # percorrer o sinal com passo no_overlap
    for i in range(0, signal.shape[0], no_overlap):
        if i + window_size < signal.shape[0]:
            segment = signal[i : i + window_size]
            segments.append(segment)  # cut segments
            t_segment.append(np.arange(i, i + window_size))  # cut time

    return segments, t_segment

In [ ]:
# função que processa o sinal CSI
# # normalização
# # segmentação
# # extração de características
def signal_processing_pipeline(
    csi_data: np.ndarray,
    normalization_mean: np.ndarray,
    normalization_abs_max: np.ndarray,
) -> np.ndarray:
    
    smoothened_data: list[np.ndarray] = []

    # normalizar o sinal
    csi_data_normalized = csi_data - normalization_mean
    csi_data_normalized = csi_data_normalized / normalization_abs_max

    # Iterate over the subsampled data and smoothen the signals
    # para cada sinal (coluna) (dos dados transpostos)
    # vamos suavizá-lo.......... PARA QUÊ??
    #for signal in csi_data_normalized.T:
    #    smoothened_signal = smooth(signal, 20)  # Subsample by a factor of 5
    #    smoothened_data.append(smoothened_signal)

    # segmentar o sinal
    segments, t_segments = segments_signal(csi_data_normalized)
    segments = np.array(segments)

    # calcular média/std de cada segmento (ao longo da linha)
    features = [np.mean(segments, axis=1), np.std(segments, axis=1)]
    features = np.array(features)

    # transpor e remodelar a matriz de características
    # mover o eixo 1 para a posição 0 e eixo 0 para a posição 1
    # o eixo 2 mantém-se
    X: np.ndarray = np.transpose(features, (1, 0, 2))

    # shape[0] é o da 1a posição
    # -1 significa "calcula automaticamente"
    X = X.reshape(X.shape[0], -1)
    return X


# função que gera as matrizes X e Y para a ESP dada
# executada 3 vezes (uma por ESP)

# Óscar: Isto é o dataset para o modelo de colunas e linhas, o outro é uma alteração desta função ou há outra?
def generate_X_Y_for_esp(
    esp_id: int, normalization_mean: np.ndarray, normalization_abs_max: np.ndarray
):
    X_total: list[np.ndarray] = []
    Y_total: list[list[int]] = []

    # para cada posição/esp
    for key in csi_data:
        if key.startswith(f"esp{esp_id}_"):
            # informação csi
            csi = csi_data[key]["csi"]
            X = signal_processing_pipeline(
                csi, normalization_mean, normalization_abs_max
            )
            X_total.append(X)

            # Extração da posição (ex: a01 → coluna=a, linha=01)
            pos = key.split("_")[1]
            col = ord(pos[0]) - ord("a") + 1
            row = int(pos[1:]) if pos[1:].isdigit() else 0
            Y_total.extend([[col, row]] * len(X))
            count += 1

    return np.concatenate(X_total), np.array(Y_total)

In [8]:
user_ids = range(0,3)
esp_ids = range(1,5)
rep_max = 3
max_user_pos = len(positions) * rep_max

count = 0
for user_id in user_ids:
   for pos in ["z00"]:
        for esp_id in esp_ids:
            for rep in range(1, rep_max + 1):
                key = f"{user_id}_{pos}_{esp_id}_{rep}"
                #print(f"Checking key: {key}")
                if key in data_files:
                    #print(f"Key {key} in data")
                    count = count + 1

print(f"Total no-presence samples: {count}")

#max_user_pos = len(positions) - 2) 

Total no-presence samples: 24


In [40]:
# subportadoras a considerar - desnecessário visto que é overwritten dentro da função ...
subcarrier_indices: list[int] = [i for i in range(2, 48)] 
csi_data: dict[str, dict[str, np.ndarray]] = {}
num_samples: int = 20

user_ids = range(0,3)
esp_ids = range(1,5)
rep_max = 3
max_user_pos = len(positions) * rep_max


X = np.array([])  # Inicializar X como um array numpy vazio
Y = np.array([])  # Inicializar Y como um array numpy vazio


for user_id in user_ids:
    for pos in positions:
        for rep in range(1, rep_max + 1):
            X_allesps = np.array([])  # Inicializar X_allesps como um array numpy 
            add = True

            for esp_id in esp_ids:
                # Stuck at repmax = 1 for now
                key = f"{user_id}_{pos}_{esp_id}_{rep}"
                
                # Check if the files corresponding to the key exist
                if key not in data_files:
                    add = False
                    continue

                # Process each file associated with the key
                file_path = data_files[key][0]

                #print(f"Processing file: {file_path}")
                magnitudes = read_and_parse_csi(file_path, 0, subcarrier_indices)

                # Normalize magnitudes per-file (mean and absolute max per subcarrier)
                if magnitudes.size == 0:
                    continue

                mean_per_subcarrier = np.mean(magnitudes, axis=0)
                abs_max_per_subcarrier = np.max(np.abs(magnitudes), axis=0)
                dados_normalizados = (magnitudes - mean_per_subcarrier) / abs_max_per_subcarrier

                # segmentar a informação em janelas de 10 amostras com sobreposição de 8
                segmentos, _ = segments_signal(dados_normalizados, window_size=10, no_overlap =2)
                seglen = len(segmentos)

                segmentos = np.array(segmentos)
                if segmentos.size == 0:
                    continue
                
                desired = num_samples * 12 if pos == "z00" else num_samples
                desired = min(desired, seglen)

                indices = np.random.choice(len(segmentos), size=desired, replace=False)
                segmentos = segmentos[indices]

                # calcular média, std e máximos de cada segmento (para cada subportadora ao longo do tempo)
                means = np.mean(segmentos, axis=1)
                stds = np.std(segmentos, axis=1)
                maxs = np.max(segmentos, axis=1)

                # concatenar as características para a ESP atual
                X_esp = np.hstack((means, stds, maxs))  # Concatenate features for the current ESP
                # print(f"X_esp shape for ESP {esp_id}: {X_esp.shape}")
            
                # conctenar as características de todas as ESPs e limitar os arrays se necessário 
                
                if X_allesps.size == 0:
                    X_allesps = X_esp
                else:
                    min_length =  min(X_esp.shape[0], X_allesps.shape[0])  # Usar o comprimento da ESP atual como referência
                
                    if X_allesps.shape[0] > min_length:
                        X_allesps = X_allesps[:min_length, :]
                    if X_esp.shape[0] > min_length:
                        X_esp = X_esp[:min_length, :]

                    # concatenar as características numa nova dimensão
                    X_allesps = np.hstack((X_allesps, X_esp))            

            if add:
                # reshape do X_allesps para intercalar as amostras das diferentes ESPs
                # cada linha corresponde a uma amostra, com as características das diferentes ESPs intercaladas
                print(f"X_allesps shape for user {user_id} position {pos}: {X_allesps.shape}")

                # armazenar os dados processados e etiquetados no dataframe final
                X = np.vstack((X, X_allesps)) if X.size else X_allesps
                
                # armazenar as etiquetas correspondentes
                if pos == "z00":  # posição de vazio
                    label = 0;
                else:
                    label = 1;

                Y = np.append(Y, [label] * X_allesps.shape[0]) 


X_allesps shape for user 0 position z00: (97, 552)
X_allesps shape for user 0 position z00: (145, 552)
X_allesps shape for user 1 position z00: (74, 552)
X_allesps shape for user 1 position z00: (240, 552)
X_allesps shape for user 1 position a00: (20, 552)
X_allesps shape for user 1 position a01: (18, 552)
X_allesps shape for user 1 position a02: (20, 552)
X_allesps shape for user 1 position a09: (20, 552)
X_allesps shape for user 1 position a10: (20, 552)
X_allesps shape for user 1 position a11: (20, 552)
X_allesps shape for user 1 position b00: (20, 552)
X_allesps shape for user 1 position b01: (20, 552)
X_allesps shape for user 1 position b02: (20, 552)
X_allesps shape for user 1 position b03: (20, 552)
X_allesps shape for user 1 position b04: (20, 552)
X_allesps shape for user 1 position b05: (20, 552)
X_allesps shape for user 1 position b06: (20, 552)
X_allesps shape for user 1 position b07: (20, 552)
X_allesps shape for user 1 position b08: (20, 552)
X_allesps shape for user 1 po

In [41]:
print(f"Final X shape: {X.shape}")
print(f"Final Y shape: {Y.shape}")

Final X shape: (2608, 552)
Final Y shape: (2608,)


In [44]:
# Split the dataset into training and testing sets
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.8, random_state=42, stratify=Y)   

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Y_train shape:", Y_train.shape)
print("Y_test shape:", Y_test.shape)

# See class distribution
(unique, counts) = np.unique(Y, return_counts=True)
print("Class distribution:", dict(zip(unique, counts)))


X_train shape: (521, 552)
X_test shape: (2087, 552)
Y_train shape: (521,)
Y_test shape: (2087,)
Class distribution: {np.float64(0.0): np.int64(890), np.float64(1.0): np.int64(1718)}


In [45]:
model = RandomForestClassifier(
	min_samples_split=4,
	min_samples_leaf=4,
	max_leaf_nodes=64,
	n_estimators=150,
	random_state=200,
	max_depth=16,
	class_weight="balanced_subsample",
	criterion="entropy",
)

model = model.fit(X_train, Y_train)

In [46]:
y_pred = model.predict(X_test)

# Note that while balanced_accuracy_score is used, if the dataset is imbalanced training could be affected.
ball_acc = balanced_accuracy_score(Y_test, y_pred)
print("Balanced Accuracy:", ball_acc)

Balanced Accuracy: 0.9668442288049031


In [47]:
for n_tree in [10, 50, 100, 150]:
    print(f"Number of trees: {n_tree}")
    model = RandomForestClassifier(
        min_samples_split=4,
        min_samples_leaf=4,
        max_leaf_nodes=64,
        n_estimators=n_tree,
        random_state=200,
        max_depth=16,
        class_weight="balanced_subsample",
        criterion="entropy",
        oob_score=True,
        
    )   
    model = model.fit(X_train, Y_train)

    y_pred = model.predict(X_test)
    ball_acc = balanced_accuracy_score(Y_test, y_pred)
    print(f"Balanced Accuracy for {n_tree} trees:", ball_acc)

Number of trees: 10
Balanced Accuracy for 10 trees: 0.9518718079673136
Number of trees: 50


c:\Users\Oscar\miniconda3\envs\isac\Lib\site-packages\sklearn\ensemble\_forest.py:611: UserWarning: Some inputs do not have OOB scores. This probably means too few trees were used to compute any reliable OOB estimates.
  warn(


Balanced Accuracy for 50 trees: 0.9622170582226762
Number of trees: 100
Balanced Accuracy for 100 trees: 0.9619034729315628
Number of trees: 150
Balanced Accuracy for 150 trees: 0.9668442288049031
